# Salon Experiment & Causal Analysis
**Part A** A/B test design + analysis (rebooking-offer experiment) | **Part B** Difference-in-differences for last quarter's change.
> Data below is SIMULATED to match the brief (22 outlets, no-shows, walk-ins). To use the real file: replace the simulation cell with `df = pd.read_csv("your_file.csv")` and re-run all cells.

In [ ]:
import numpy as np, pandas as pd, json, matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(7); Z = stats.norm.ppf; R = {}

## 1. Data (simulated) and cleaning rules

In [ ]:
n = 10200
df = pd.DataFrame({"client_id": np.arange(n), "variant": rng.choice(["control", "treatment"], n),
    "outlet": rng.integers(1, 23, n), "client_type": rng.choice(["new", "returning"], n, p=[.35, .65]),
    "service": rng.choice(["Hair", "Skin", "Spa", "Bridal"], n, p=[.45, .25, .22, .08]),
    "pre_spend": rng.lognormal(7.1, .45, n).round()})
t = (df.variant == "treatment").astype(int).values; new = (df.client_type == "new").astype(int).values
df["retained_60d"] = (rng.random(n) < .30 + .22*(1-new) + t*(.07*new + .02*(1-new))).astype(int)
df["no_show"] = (rng.random(n) < .17 - .01*t).astype(int)
df["rev_per_visit"] = (.75*df.pre_spend + rng.normal(0, 550, n) - 20*t).clip(lower=0).round()
# cleaning demo on raw appointment rows: no-shows tagged as cancellations, outlet-specific service names, walk-ins w/o client ID
raw = pd.DataFrame({"service_name": ["Hair Cut", "Haircut - Men", "HAIRCUT", "Facial (Gold)", "Gold Facial"],
    "status": ["cancelled", "completed", "cancelled", "completed", "completed"],
    "reason": ["no show", "", "customer cancelled", "", ""], "client_id": [101, None, 103, 104, 105]})
canon = lambda s: "Haircut" if "hair" in s.lower() else "Facial" if "facial" in s.lower() else s
raw["service"] = raw.service_name.map(canon)
raw.loc[(raw.status == "cancelled") & raw.reason.str.contains("no show"), "status"] = "no_show"
raw["walk_in"] = raw.client_id.isna()   # keep for revenue, exclude from client-level retention
print(raw[["service", "status", "walk_in"]])

## 2. Power analysis (primary metric: 60-day retention)

In [ ]:
def n_arm(p1, d, a=.05, pw=.8):
    p2 = p1 + d; pb = (p1 + p2)/2
    return int(np.ceil((Z(1-a/2)*np.sqrt(2*pb*(1-pb)) + Z(pw)*np.sqrt(p1*(1-p1) + p2*(1-p2)))**2 / d**2))
base, mde, weekly = .44, .03, 1000
na = n_arm(base, mde); nb = int(np.ceil(na*1.1)); wk = int(np.ceil(2*nb/weekly))
R["power"] = dict(base=base, mde=mde, n_arm=na, n_arm_buffer=nb, weekly=weekly, weeks=wk, followup_days=60)
R["power_curve"] = {f"{d:.3f}": n_arm(base, d) for d in [.02, .025, .03, .04, .05, .06]}
print(R["power"])

## 3. Randomisation checks: sample ratio mismatch + covariate balance

In [ ]:
obs = df.variant.value_counts().reindex(["control", "treatment"]).values
R["srm"] = dict(control=int(obs[0]), treatment=int(obs[1]), p=float(stats.chisquare(obs).pvalue))
bal = {c: float(stats.chi2_contingency(pd.crosstab(df.variant, df[c]))[1]) for c in ["client_type", "service", "outlet"]}
a, b = df.pre_spend[t == 1], df.pre_spend[t == 0]
smd = (a.mean() - b.mean())/np.sqrt((a.var() + b.var())/2)
bal["pre_spend"] = float(stats.ttest_ind(a, b, equal_var=False).pvalue)
R["balance"] = dict(p=bal, smd_pre_spend=float(smd)); print(R["srm"], R["balance"])

## 4. Effect estimates with 95% CI (Welch / bootstrap)

In [ ]:
def diff(y, g, alpha=.05):
    a, b = y[g == 1], y[g == 0]; d = a.mean() - b.mean()
    se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b)); z = Z(1 - alpha/2)
    return dict(c=float(b.mean()), t=float(a.mean()), d=float(d), lo=float(d - z*se), hi=float(d + z*se), se=float(se),
                p=float(2*(1 - stats.norm.cdf(abs(d/se)))))
def boot(y, g, B=2000):
    a, b = y[g == 1], y[g == 0]
    return np.percentile([rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(B)], [2.5, 97.5]).tolist()
R["eff"] = {m: diff(df[m].values, t) for m in ["retained_60d", "no_show", "rev_per_visit"]}
R["boot_rev"] = boot(df.rev_per_visit.values, t); print(R["eff"], R["boot_rev"])

## 5. Segments (Bonferroni-adjusted CIs for 6 looks)

In [ ]:
seg = {}
for col in ["client_type", "service"]:
    for v in df[col].unique():
        m = (df[col] == v).values; seg[f"{col}={v}"] = diff(df.retained_60d.values[m], t[m], alpha=.05/6) | {"n": int(m.sum())}
R["seg"] = seg
for k, v in seg.items(): print(k, round(v["d"]*100, 1), round(v["lo"]*100, 1), round(v["hi"]*100, 1))

## 6. CUPED variance reduction (revenue per visit, covariate = pre-experiment spend)

In [ ]:
x, y = df.pre_spend.values, df.rev_per_visit.values
theta = np.cov(y, x)[0, 1]/x.var(ddof=1); y_adj = y - theta*(x - x.mean())
raw_e, adj_e = diff(y, t), diff(y_adj, t); vr = 1 - (adj_e["se"]/raw_e["se"])**2
R["cuped"] = dict(theta=float(theta), corr=float(np.corrcoef(x, y)[0, 1]), raw=raw_e, adj=adj_e, var_red=float(vr)); print(R["cuped"])

## 7. Last quarter's change: difference-in-differences (12 treated outlets vs 10 control outlets)

In [ ]:
months = pd.period_range("2025-07", "2026-09", freq="M"); post = (months >= pd.Period("2026-07"))
tr = np.arange(1, 23) <= 12
season = np.array([0, .03, .05, .10, .08, .12, .02, -.02, -.03, 0, .04, .02, .05, .10, .14])
Y = (np.log(15) + rng.normal(0, .25, 22))[:, None] + season + .005*np.arange(15) + rng.normal(0, .03, (22, 15)) + .04*tr[:, None]*post
naive = Y[tr][:, post].mean() - Y[tr][:, ~post].mean()
dl = Y[:, post].mean(1) - Y[:, ~post].mean(1); a, b = dl[tr], dl[~tr]; did = a.mean() - b.mean()
se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
dfw = se**4/((a.var(ddof=1)/len(a))**2/(len(a)-1) + (b.var(ddof=1)/len(b))**2/(len(b)-1)); q = stats.t.ppf(.975, dfw)
pt = lambda v: float(np.expm1(v)*100)
R["did"] = dict(naive_pct=pt(naive), did_pct=pt(did), lo_pct=pt(did - q*se), hi_pct=pt(did + q*se), p=float(2*stats.t.sf(abs(did/se), dfw)))
# parallel trends: per-outlet pre-period slopes, treated vs control; plus placebo DiD (fake launch 3 months before real one)
sl = np.array([stats.linregress(np.arange(12), Y[i, :12]).slope for i in range(22)]); s1, s0 = sl[tr], sl[~tr]
ds = s1.mean() - s0.mean(); se_s = np.sqrt(s1.var(ddof=1)/len(s1) + s0.var(ddof=1)/len(s0))
R["pretrend"] = dict(slope_diff=float(ds), lo=float(ds - 2.08*se_s), hi=float(ds + 2.08*se_s), p=float(stats.ttest_ind(s1, s0, equal_var=False).pvalue))
fp = np.arange(12) >= 9; dp = Y[:, :12][:, fp].mean(1) - Y[:, :12][:, ~fp].mean(1); ap, bp = dp[tr], dp[~tr]
pl = ap.mean() - bp.mean(); se_p = np.sqrt(ap.var(ddof=1)/len(ap) + bp.var(ddof=1)/len(bp))
R["placebo"] = dict(pct=pt(pl), lo=pt(pl - 2.08*se_p), hi=pt(pl + 2.08*se_p), p=float(stats.ttest_ind(ap, bp, equal_var=False).pvalue))
print(R["did"], R["pretrend"], R["placebo"])

## 8. Charts

In [ ]:
lab = ["Overall"] + [k.replace("client_type=", "Client: ").replace("service=", "Service: ") for k in seg]; est = [R["eff"]["retained_60d"]] + list(seg.values())
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar([e["d"]*100 for e in est], range(len(est)), xerr=[[(e["d"]-e["lo"])*100 for e in est], [(e["hi"]-e["d"])*100 for e in est]], fmt="o", capsize=3)
ax.axvline(0, color="grey", ls="--"); ax.set_yticks(range(len(est))); ax.set_yticklabels(lab); ax.invert_yaxis()
ax.set_xlabel("Effect on 60-day retention (percentage points, 95% CI)"); plt.tight_layout(); plt.savefig("figs/forest.png", dpi=150); plt.show()
g = np.exp(Y)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8)); mm = months.strftime("%b %y")
ax[0].plot(mm, g[tr].mean(0), label="Treated outlets"); ax[0].plot(mm, g[~tr].mean(0), label="Control outlets")
ax[0].axvline(11.5, color="grey", ls="--"); ax[0].set_ylabel("Revenue per outlet (Rs lakh)"); ax[0].legend()
gap = Y[tr].mean(0) - Y[~tr].mean(0); gap -= gap[:12].mean()
ax[1].bar(mm, gap*100, color=["C0"]*12 + ["C1"]*3); ax[1].axhline(0, color="k", lw=.8); ax[1].set_ylabel("Treated - control gap (% vs pre-avg)")
for a_ in ax: a_.tick_params(axis="x", rotation=90, labelsize=7)
plt.tight_layout(); plt.savefig("figs/did.png", dpi=150); plt.show()
plt.figure(figsize=(5.5, 3.5)); plt.plot([float(k)*100 for k in R["power_curve"]], list(R["power_curve"].values()), "o-")
plt.xlabel("Minimum detectable effect (pp)"); plt.ylabel("Clients per arm"); plt.tight_layout(); plt.savefig("figs/power.png", dpi=150); plt.show()

In [ ]:
json.dump(R, open("results.json", "w"), indent=1)